# D · 원하람 — Sentence-BERT 문체 융합 (D-3 ~ D-4)

`haram` 브랜치. Accelerator GPU T4 x2, Internet On.

## 1. 저장소·환경

In [ ]:
%cd /kaggle/working
!rm -rf LLM-name-tagging
!git clone -b haram https://github.com/seoultech-Machine-Learning-26-2/LLM-name-tagging.git
%cd /kaggle/working/LLM-name-tagging
!git log -1 --oneline

import os
assert os.path.exists('experiments/D_fusion.py'), 'haram 브랜치가 push 됐는지 확인'

import torch
assert torch.cuda.is_available(), 'Settings에서 Accelerator를 GPU로 바꾸고 다시 실행'
print(torch.__version__, torch.cuda.get_device_name(0))

## 1-1. 패키지·한글 폰트

In [ ]:
!pip -q install -U sentence-transformers
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
!fc-cache -f > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.font_manager as fm
print('NanumGothic:', any('Nanum' in f.name for f in fm.fontManager.ttflist))

## 2. 융합 실험 (D-3 ~ D-4)

의미 / 문체 / 융합 × 원본 / 질문중심화 = 6개 분기, 각 seed 3개.

In [ ]:
!python -u -W ignore -m experiments.D_fusion

## 3. 결과

In [ ]:
import pandas as pd
from IPython.display import Image, display as d
pd.set_option('display.width', 220)
B = 'results/course/D/fusion/'

res = pd.read_csv(B+'val_results.csv')
cols = ['run_id','config','seed','n_features','n_trainable_params','best_epoch',
        'train_model_BA','model_BA','accuracy','macro_f1','log_loss','ece',
        'sibling_acc','company_BA','fit_seconds']
display(res[[c for c in cols if c in res.columns]].round(4))

### 3-1. 분기별 요약 (seed 3개)

In [ ]:
g = res.groupby('config')[['train_model_BA','model_BA','macro_f1','company_BA',
                           'n_trainable_params','best_epoch']].agg(['mean','std'])
order = ['semantic','semantic_centered','style','style_centered','fusion','fusion_centered']
display(g.reindex(order).round(4))
d(Image(B+'fig_branch_comparison.png'))

### 3-2. 혼동행렬 (융합, seed 42)

In [ ]:
d(Image(B+'fig_confusion_val.png'))

### 3-3. 보정·형제 혼동·모델별 재현율

In [ ]:
print('확신도 구간별 (gap > 0 이면 과신)')
display(pd.read_csv(B+'val_calibration.csv'))
print('오분류 중 같은 개발사 비율')
display(pd.read_csv(B+'val_sibling_errors.csv'))
print('모델별 재현율')
display(pd.read_csv(B+'val_per_class.csv'))

### 3-3b. 질문 중심화 효과

In [ ]:
piv = (res.pivot_table(index='feature_set', columns='centering',
                      values='model_BA', aggfunc='mean')
          .reindex(['semantic','style','fusion'])
          .reindex(columns=['raw','centered']))
piv['차이'] = piv.centered - piv.raw
display(piv.round(4))

### 3-3c. 개발사별 분해

In [ ]:
display(pd.read_csv(B+'val_company_breakdown.csv').round(4))

### 3-4. 오분류 5개

In [ ]:
err = pd.read_csv(B+'val_errors_seed42.csv')
for _, r in err.iterrows():
    print(f"[{r.id}] {r.true_model} -> {r.pred_model} "
          f"(p={r.pred_prob:.3f}, 정답 p={r.true_prob:.3f}, 같은 개발사={r.same_company})")
    print('   ', r.text_head[:200], '\n')

### 3-5. 고전 모델과 비교

In [ ]:
g = res.groupby('config')[['train_model_BA','model_BA','macro_f1','company_BA',
                           'n_trainable_params','best_epoch']].agg(['mean','std'])
order = ['semantic','semantic_centered','style','style_centered','fusion','fusion_centered']
display(g.reindex(order).round(4))
d(Image(B+'fig_branch_comparison.png'))

## 4. 다른 인코더 (선택)

In [ ]:
# !python -u -W ignore -m experiments.D_fusion \
#     --encoder sentence-transformers/all-mpnet-base-v2 \
#     --out-dir results/course/D/fusion_mpnet

## 5. 결과 저장

In [ ]:
!zip -qr /kaggle/working/D_fusion_results.zip results/course/D preds/val_D_* \
     -x 'results/course/D/embeddings/*' 2>/dev/null
!ls -lh /kaggle/working/D_fusion_results.zip
!unzip -l /kaggle/working/D_fusion_results.zip | tail -25

## 6. test 1회 (팀 합의 후에만)

In [ ]:
# !python -u -W ignore -m experiments.D_fusion --final fusion